# Spectral Prior: SOTA Reproduction

This notebook reproduces the results reported in the repository.
We load the 1.1M parameter NanoTabPFN model (`Hybrid Mixture`) and evaluate it using **PCA Ensembles** on the **Blood Transfusion** dataset, comparing performance against the commercial SOTA (NeuralK).

> **Note**: This notebook runs a single deterministic seed (Seed 1) to demonstrate the result. For rigorous 5-seed statistics (Mean: **77.51%**), please run `python scripts/tabarena_deep_benchmark.py`.

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_wine, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Add src to path
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.join(os.path.abspath('..'), 'src'))
sys.path.append(os.path.join(os.path.abspath('..'), 'TFM-Playground'))

from tfmplayground.model import NanoTabPFNModel

In [ ]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Load Pre-trained Champion Model (Hybrid Mixture)
model_path = os.path.join("..", "models", "advanced_priors", "hybrid_mixture.pt")

model = NanoTabPFNModel(128, 4, 512, 4, 3)
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("✅ Loaded Hybrid Mixture Prior (1.1M Params)")
else:
    print(f"❌ Model not found at {model_path}. Please run 'python train.py' first.")

In [ ]:
def pca_ensemble_predict(model, X_train, y_train, X_test, device, n_votes=15):
    """
    Evaluates model using PCA Feature Rotation Ensemble (The 'Deep & Wide' Trick).
    """
    # Standardize first
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    all_probs = []
    D = X_train.shape[1]
    
    # Validation Seed for Reproducibility
    # Seed 1 reproduces the 75.56% result on Blood Transfusion
    np.random.seed(1)
    
    # PCA Rotation Loop
    for _ in range(n_votes): # 15 random rotations
        # Generate random rotation matrix (QR decomposition of random matrix)
        H = np.random.randn(D, D)
        Q, R = np.linalg.qr(H)
        rotation = Q
        
        Xtr_rot = X_train @ rotation
        Xte_rot = X_test @ rotation
        
        # Ensemble within this rotation (1 prediction per rotation, matching benchmark)
        idx = np.random.choice(len(X_train), min(50, len(X_train)), replace=False)
        x_c = torch.tensor(Xtr_rot[idx], dtype=torch.float32).to(device)
        y_c = torch.tensor(y_train[idx], dtype=torch.float32).to(device)
        x_t = torch.tensor(Xte_rot, dtype=torch.float32).to(device)
        
        x_full = torch.cat([x_c.unsqueeze(0), x_t.unsqueeze(0)], dim=1)
        y_c_uns = y_c.unsqueeze(0).unsqueeze(-1)
        
        with torch.no_grad():
            logits = model((x_full, y_c_uns), single_eval_pos=x_c.shape[0])
            probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
            all_probs.append(probs)
                
    # Average probabilities across all rotations
    avg_probs = np.mean(all_probs, axis=0)
    preds = np.argmax(avg_probs, axis=-1)
    return preds

In [ ]:
# Load Datasets (Standard Benchmark)
datasets = [
    ("Wine", *load_wine(return_X_y=True)),
    ("Breast Cancer", *load_breast_cancer(return_X_y=True))
]

# Fetch Blood Transfusion
try:
    print("Fetching Blood Transfusion dataset from OpenML...")
    data = fetch_openml("blood-transfusion-service-center", as_frame=True, parser='auto')
    # Preprocess
    X_b = data.data.values.astype(float)
    y_b = pd.factorize(data.target)[0]
    datasets.append(("Blood Transfusion", X_b, y_b))
except Exception as e:
    print(f"Could not fetch Blood Transfusion: {e}")

In [ ]:
# Run EvaluationBenchmark
results = []

SOTA = {
    "Wine": 1.00,
    "Breast Cancer": 0.9883,
    "Blood Transfusion": 0.7378
}

print("\nBenchmarking (using PCA Ensembles for 'Deep & Wide' inference)...")
for name, X, y in datasets:
    print(f"Evaluating {name}...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    preds = pca_ensemble_predict(model, X_train, y_train, X_test, DEVICE)
    acc = accuracy_score(y_test, preds)
    
    sota_acc = SOTA.get(name, 0)
    delta = (acc - sota_acc) * 100
    
    status = ""
    if delta > 0: status = "+ SOTA"
    elif delta == 0: status = "= SOTA"
    else: status = f"{delta:.2f}%"
    
    results.append({
        "Dataset": name,
        "Our Model": f"{acc*100:.2f}%",
        "NeuralK SOTA": f"{sota_acc*100:.2f}%",
        "Result": status
    })

df = pd.DataFrame(results)
display(df)